30일이 지난 과거 데이터를 Azure Blob Storage(콜드 스토리지)로 백업

In [ ]:
import sys, re
from datetime import datetime, timedelta, timezone
 
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG, BASE_PATH, ARCHIVE_CONTAINER, SECRET_SCOPE

In [ ]:
# blob 접근 인증
spark.conf.set(
    f"fs.azure.account.key.dt4team1blob.dfs.core.windows.net",
    dbutils.secrets.get(scope=f"{SECRET_SCOPE}", key="blob-storage-key")
)

In [ ]:
SCHEMA = "gold"
 
# BASE_PATH에서 storage account 파싱
# abfss://raw@dt4team1blob.dfs.core.windows.net/raw
STORAGE_ACCOUNT = re.search(r"@(.+?)\.dfs", BASE_PATH).group(1)

# 아카이브 저장 경로: archive/{오늘날짜}/{테이블명}
now          = datetime.now(timezone.utc)
today_str    = now.strftime("%Y%m%d")
yesterday    = (now - timedelta(days=1)).strftime("%y%m%d")  # 6자리: 260702
ARCHIVE_PATH = f"abfss://{ARCHIVE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{yesterday}"
DATE_PATTERN = re.compile(r"_(\d{6})$")  # 6자리 패턴

In [ ]:
tables = spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{SCHEMA}`").collect()
 
for row in tables:
    name = row["tableName"]
    m = DATE_PATTERN.search(name)
 
    if not m or m.group(1) != yesterday:
        print(f"[SKIP] {name}")
        continue
 
    dest = f"{ARCHIVE_PATH}/{name}"
 
    # archive로 이전 후 원본 삭제
    (spark.read.format("delta")
         .table(f"`{CATALOG}`.`{SCHEMA}`.`{name}`")
         .write.format("delta")
         .mode("overwrite")
         .save(dest))
 
    spark.sql(f"DROP TABLE `{CATALOG}`.`{SCHEMA}`.`{name}`")
    print(f"[OK] {name} → {dest} | 원본 삭제")

In [ ]:
%sql
drop table dt4_team1_databricks.gold.ml_dataset_radius_all_260703